# Figures for the blog essay

Generates the figures called out in the
[Spec 03 outreach plan](../docs/specs/v0.3/03-outreach-and-essay.md).
All output is seeded and deterministic; re-running produces
byte-identical PNGs.

Output directory: ``notebooks/figures/`` (gitignored). The
maintainer re-runs this notebook before publishing the essay
and copies the PNGs into the blog post manually.

## Setup

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nidus

rng = np.random.default_rng(seed=20251101)

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

ds = nidus.load()

## Figure 1 — Tier distribution by subsystem

Stacked bar showing how the four tiers are distributed across
the subsystems. The visual point: data density is uneven,
and the tier system surfaces where the dataset is well-grounded
versus where it is provisional.

In [ ]:
SUBSYSTEMS = list(ds.subsystems())
TIERS = ["A", "B", "C", "D"]
COLORS = {"A": "#2ca02c", "B": "#1f77b4", "C": "#ff7f0e", "D": "#d62728"}

counts = {(s, t): 0 for s in SUBSYSTEMS for t in TIERS}
for p in ds:
    counts[(p.subsystem, p.tier)] += 1

fig, ax = plt.subplots(figsize=(9, 5))
bottoms = [0] * len(SUBSYSTEMS)
for tier in TIERS:
    heights = [counts[(s, tier)] for s in SUBSYSTEMS]
    ax.barh(SUBSYSTEMS, heights, left=bottoms, color=COLORS[tier], label=f"Tier {tier}")
    bottoms = [b + h for b, h in zip(bottoms, heights)]

ax.set_xlabel("Number of parameters")
ax.set_title("Tier distribution by subsystem")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES / "fig1_tier_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

## Figure 2 — Worked-example timeline (Mahendru 2014)

A diagram showing how a single peer-reviewed paper flows
through nidus: original publication → parameter extraction →
tier assignment → use in the dataset → exposure to dashboard
and downstream computations.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis("off")

STAGES = [
    ("Mahendru et al.\n2014\n(J. Hypertension)", "#cdd9f0"),
    ("Extracted\nvalues +\ncontext", "#cdd9f0"),
    ("Tier B\nassigned\n(verified)", "#cde8d2"),
    ("Stored in\ndataset/\nparameters/", "#cdd9f0"),
    ("Resolved by\nnidus.load()\nfor downstream\ncomputation", "#f0e3cd"),
]
N = len(STAGES)
margin = 0.4
width = (10 - 2 * margin - (N - 1) * 0.4) / N

for i, (label, color) in enumerate(STAGES):
    x = margin + i * (width + 0.4)
    ax.add_patch(plt.Rectangle((x, 0.8), width, 1.4, facecolor=color, edgecolor="#444"))
    ax.text(x + width / 2, 1.5, label, ha="center", va="center", fontsize=9)
    if i < N - 1:
        arrow_x = x + width + 0.05
        ax.annotate(
            "", xy=(arrow_x + 0.3, 1.5), xytext=(arrow_x, 1.5),
            arrowprops=dict(arrowstyle="->", color="#444", lw=1.4),
        )

ax.text(5, 0.3, "Same model applies to every other Tier-A/B parameter in the dataset.",
        ha="center", fontsize=9, color="#666")
ax.set_title("From paper to parameter to downstream use", pad=14)
plt.tight_layout()
plt.savefig(FIGURES / "fig2_worked_example_timeline.png", dpi=180, bbox_inches="tight")
plt.show()

## Figure 3 — Citation usage histogram

How many parameters depend on each citation? The shape of this
distribution tells you how concentrated the dataset's
provenance is. A few load-bearing papers carry most of the
weight; a long tail of supporting papers contribute one or two
parameters each.

Replaces the Spec-03 "dashboard screenshot" placeholder (the
screenshot is captured manually from the running dashboard, not
from this notebook).

In [ ]:
uses = Counter(c.key for p in ds for c in p.citations)
usage_counts = sorted(uses.values(), reverse=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(usage_counts)), usage_counts, color="#1f77b4", edgecolor="white")
ax.set_xlabel("Citation (sorted by usage)")
ax.set_ylabel("Number of parameters citing this work")
ax.set_title(f"Citation usage across {len(usage_counts)} cited works")
ax.axhline(np.mean(usage_counts), color="#444", linestyle="--", linewidth=1,
           label=f"mean = {np.mean(usage_counts):.1f}")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig3_citation_usage_histogram.png", dpi=180, bbox_inches="tight")
plt.show()

## Done

The three PNGs are now in ``notebooks/figures/`` and can be
embedded in the blog essay. Re-run this notebook any time the
dataset changes; the figures will update deterministically.